# Registration Datasets

Registration datasets pair two scans of the same scene with the rigid transform between them: the source `pos`, the reference `ref_pos` and the $4 \times 4$ `transform` moving the source into the reference frame.

![A 3DMatch pair in its two frames and aligned by the ground truth](../assets/datasets/registration.png)

<div class="tp-catalog" markdown>

| Dataset                                             | Pairs                            | Overlap      |
| --- | --- | --- |
| [`ThreeDMatch`](../api/datasets/threedmatch.md)     | 20 642 train / 1 331 val / 1 623 test | above 30%  |
| [`ThreeDMatch`](../api/datasets/threedmatch.md) (3DLoMatch) | 1 781 test                | 10% to 30%   |
| [`KITTIOdometry`](../api/datasets/kittiodometry.md) | 1 358 train / 180 val / 555 test | scans 10 m apart |
| [`ModelNet40Registration`](../api/datasets/modelnet.md) | 2 148 test pairs of 32 shape classes | 70% crops |

</div>


## Load a dataset

The dataset takes its data directory as `root`. Pass `download=True` when the data is not there yet: the fragments come from the PREDATOR release and the pair lists from the GeoTransformer repository, and the first use converts the fragments to a `float32` cache.

```text
data/
└── ThreeDMatch/
    ├── raw/
    │   ├── indoor/
    │   │   ├── train/<scene>/cloud_bin_*.pth
    │   │   └── test/<scene>/cloud_bin_*.pth
    │   └── metadata/
    │       ├── train.pkl
    │       ├── val.pkl
    │       └── benchmarks/{3DMatch,3DLoMatch}/<scene>/{gt.log,gt.info,gt_overlap.log}
    └── processed/
        ├── train/<scene>/cloud_bin_*.npy
        ├── test/<scene>/cloud_bin_*.npy
        └── pairs/{train,val,3DMatch,3DLoMatch}.json
```

In [ ]:
from torch_pointcloud.config import DATA_DIR
from torch_pointcloud.datasets import ThreeDMatch

dataset = ThreeDMatch(root=DATA_DIR, split="test", benchmark="3DMatch", download=True)
print(f"Pairs: {len(dataset)}")
print({k: tuple(v.shape) if hasattr(v, "shape") else v for k, v in dataset[0].items()})

# Pairs: 1623
# {'pos': (19082, 3), 'ref_pos': (18977, 3), 'transform': (4, 4), 'overlap': (), 'frame': '1', 'ref_frame': '0', 'scene': '7-scenes-redkitchen', 'information': (6, 6)}

`split="test"` pairs the 8 benchmark scenes as the 3DMatch or the 3DLoMatch benchmark; `split="train"` and `split="val"` are the PREDATOR training pairs.

## What a sample holds

| Key           | Shape    | Description                                                        |
| ------------- | -------- | ------------------------------------------------------------------ |
| `pos`         | $(N, 3)$ | Source fragment, voxel-downsampled at 2.5 cm                       |
| `ref_pos`     | $(M, 3)$ | Reference fragment                                                 |
| `transform`   | $(4, 4)$ | Transform of the source onto the reference                         |
| `overlap`     | scalar   | Overlap ratio of the pair                                          |
| `information` | $(6, 6)$ | Information matrix of the ground truth, test split only            |
| `frame`       | str      | Source fragment id                                                 |
| `ref_frame`   | str      | Reference fragment id                                              |
| `scene`       | str      | Scene name                                                         |

## Batch the pairs

The two clouds of a pair have different sizes, so the reference collates as a `cat_keys` entry and arrives with its own `batch_ref_pos` index; the per-pair matrices stack.

In [ ]:
from torch_pointcloud.utils.data import DataKeys, PointCloudDataLoader

dataloader = PointCloudDataLoader(
    dataset,
    batch_size=1,
    cat_keys=[DataKeys.REF_POS],
    stack_keys=[DataKeys.TRANSFORM, DataKeys.INFORMATION],
)

data = next(iter(dataloader))
print(f"Pos shape: {tuple(data['pos'].shape)}")
print(f"Ref pos shape: {tuple(data['ref_pos'].shape)}")
print(f"Transform shape: {tuple(data['transform'].shape)}")
print(f"Information shape: {tuple(data['information'].shape)}")

# Pos shape: (19082, 3)
# Ref pos shape: (18977, 3)
# Transform shape: (1, 4, 4)
# Information shape: (1, 6, 6)

`batch[j]` names the pair of source point `j` and `batch_ref_pos[k]` the pair of reference point `k`; the registration models take both.

## Check the ground truth

`transform_points` applies a transform to a cloud. Moved by the ground truth, the source lands on the reference wherever the two fragments overlap.

In [ ]:
import torch

from torch_pointcloud.ops.geometry import transform_points

sample = dataset[0]
aligned = transform_points(sample["pos"], sample["transform"])
nearest = torch.cdist(aligned, sample["ref_pos"]).min(dim=1).values
print(f"Overlap {sample['overlap']:.2f}, points within 5 cm of the reference: {(nearest < 0.05).float().mean():.2f}")

# Overlap 0.80, points within 5 cm of the reference: 0.82

## LiDAR pairs

`KITTIOdometry` pairs two velodyne scans of one KITTI odometry sequence at least 10 m apart, with the ICP-refined ground truth of the PREDATOR protocol. The scans are read as released; the benchmark voxelizes them at 30 cm, so pass that `Voxelize` on both clouds as the dataset transform. The pair lists download; the velodyne scans come from the KITTI website into `KITTIOdometry/raw/sequences/`.

In [ ]:
import torch_pointcloud.transforms as T
from torch_pointcloud.datasets import KITTIOdometry

voxelize = T.Compose(
    [
        T.Voxelize(pos_key="pos", pos_reduce="mean", size=0.3, origin="min"),
        T.Voxelize(pos_key="ref_pos", pos_reduce="mean", size=0.3, origin="min"),
    ]
)
scans = KITTIOdometry(root=DATA_DIR, split="test", transform=voxelize, download=True)
sample = scans[0]
print(f"Pairs: {len(scans)}")
print({k: tuple(v.shape) if hasattr(v, "shape") else v for k, v in sample.items()})

# Pairs: 555
# {'pos': (28847, 3), 'ref_pos': (30290, 3), 'transform': (4, 4), 'frame': '000000', 'ref_frame': '000014', 'sequence': '08'}

## Synthetic pairs

`ModelNet40Registration` draws the RPM-Net benchmark pairs from the `ModelNet40Hdf5` shapes: a random rigid transform separates two copies of a shape, each is cropped by a random plane to 70% of its points, resampled to 717 points and jittered. The test pairs are seeded by their index, so they are the same at every read.

In [ ]:
from torch_pointcloud.datasets import ModelNet40Registration

pairs = ModelNet40Registration(root=DATA_DIR, train=False, download=True)
sample = pairs[0]
print(f"Pairs: {len(pairs)}")
print({k: tuple(v.shape) for k, v in sample.items()})

# Pairs: 2148
# {'pos': (717, 3), 'normal': (717, 3), 'ref_pos': (717, 3), 'ref_normal': (717, 3), 'transform': (4, 4), 'label': ()}